# 06 · Partitioning — Splitting the Keyspace

**Part 1 · Data-Intensive Applications** · *Prerequisites: 03, 05* · *Book mapping: DDIA ch. 6*

Replication (nb 05) copies *all* the data to several nodes; once data or write load outgrows
one machine, you must also **partition** (shard): split the keyspace so each node owns a
slice. Every large system does both at once — each partition is itself replicated.

The whole game is one word: **balance**. A partitioning scheme succeeds when data *and* load
spread evenly, and it fails through **skew** — one partition holding too much data or, worse,
one **hot spot** taking too much traffic, so that ten machines deliver the throughput of two.

**Learning objectives.**

1. Implement the two fundamental schemes — partition by **key range** and by **hash of key**
   — and demonstrate what each buys and each breaks.
2. Show why naive `hash % N` makes rebalancing catastrophic, then build **consistent hashing
   with virtual nodes** and measure both problems disappear.
3. Reproduce a hot-key workload and mitigate it with key salting.
4. Build both kinds of secondary index — **local** (document-partitioned) and **global**
   (term-partitioned) — and count the round trips that separate them.

In [ ]:
import hashlib
from bisect import bisect_right

import matplotlib.pyplot as plt
import numpy as np

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=6)

def stable_hash(s: str) -> int:
    """Deterministic 64-bit hash. (Python's builtin hash() is salted per process
    - using it for placement would shuffle your cluster on every restart.)"""
    return int.from_bytes(hashlib.blake2b(s.encode(), digest_size=8).digest())

## 1. Two ways to slice a keyspace

> **Key-range partitioning**: sort keys; each partition owns a contiguous range (like
> encyclopedia volumes A–F, G–M, ...). Range scans are cheap — neighbors live together.
> Used by HBase, and by every SSTable file you built in nb 03.

> **Hash partitioning**: place each key by `hash(key)`. The hash scatters similar keys
> uniformly — balance is nearly automatic — but neighbors are strangers now: a range scan
> must ask *every* partition. Used by Dynamo/Cassandra-style stores.

The classic failure demonstrates the choice. An events table keyed by timestamp, key-range
partitioned: *every* insert has "now" as its key prefix, so every insert lands in the last
partition. Ten nodes, one working:

In [ ]:
N_PARTS = 8
# 20,000 writes arriving in time order over one day:
timestamps = [f"2026-08-30T{h:02d}:{m:02d}:{s:02d}"
              for h, m, s in zip(rng.integers(0, 24, 20_000),
                                 rng.integers(0, 60, 20_000),
                                 rng.integers(0, 60, 20_000))]
timestamps.sort()

def range_partition(key: str, boundaries: list[str]) -> int:
    """Partition = which range the key falls in (boundaries are sorted split points)."""
    return bisect_right(boundaries, key)

def hash_partition(key: str, n: int) -> int:
    return stable_hash(key) % n

# Boundaries were chosen when the table was created, based on YESTERDAY's data:
boundaries = [f"2026-08-29T{h:02d}:00:00" for h in range(3, 24, 3)]

range_load = np.bincount([range_partition(t, boundaries) for t in timestamps],
                         minlength=N_PARTS)
hash_load = np.bincount([hash_partition(t, N_PARTS) for t in timestamps],
                        minlength=N_PARTS)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
axes[0].bar(range(N_PARTS), range_load, color="#c62828")
axes[0].set_title("key-range on timestamps: one hot partition")
axes[1].bar(range(N_PARTS), hash_load, color="#2e7d32")
axes[1].set_title("hash partitioning: balanced")
for ax in axes:
    ax.set_xlabel("partition")
axes[0].set_ylabel("writes")
plt.show()

assert range_load[-1] == len(timestamps), \
    "monotonically increasing keys: 100% of writes hit the LAST range partition"
assert hash_load.max() / hash_load.min() < 1.2, "hashing balances within ~20%"

But hashing pays for its balance elsewhere — the range query "events between 14:00 and
14:05" now touches every partition:

In [ ]:
lo, hi = "2026-08-30T14:00:00", "2026-08-30T14:05:00"

# Key-range: the query maps to ONE partition (or a few adjacent ones).
parts_touched_range = {range_partition(k, boundaries) for k in (lo, hi)}
# Hash: matching keys are scattered - EVERY partition must be searched.
parts_touched_hash = {hash_partition(t, N_PARTS) for t in timestamps if lo <= t <= hi}

print(f"range query touches {len(parts_touched_range)} partition(s) under key-range, "
      f"{len(parts_touched_hash)} under hashing (scatter-gather)")
assert len(parts_touched_range) == 1
assert len(parts_touched_hash) == N_PARTS, "hashing scatters ranges across everyone"

The practical synthesis (Cassandra's design): a **compound key** — hash the first part to
pick the partition, keep the rest sorted *within* the partition:
`(hash(sensor_id), timestamp)`. Balance across sensors, cheap time-ranges per sensor.
Exercise 3 explores it.

## 2. Rebalancing — and why `hash % N` is a trap

Clusters change: disks fill, nodes join and die (nb 05's failovers). **Rebalancing** moves
partitions between nodes, and the constraint is brutal: move *as little as possible*, because
moving data consumes the same disks and NICs that serve traffic.

The obvious scheme `node = hash(key) % N` fails exactly here — change N and almost every
key's assignment changes:

In [ ]:
keys = [f"user:{i:06d}" for i in range(30_000)]

def moved_fraction(assign_before: dict, assign_after: dict) -> float:
    return float(np.mean([assign_before[k] != assign_after[k] for k in keys]))

before = {k: stable_hash(k) % 5 for k in keys}
after = {k: stable_hash(k) % 6 for k in keys}          # added ONE node
mod_moved = moved_fraction(before, after)
print(f"mod-N, 5 -> 6 nodes: {mod_moved:.0%} of ALL keys must move")
assert mod_moved > 0.7, "adding one node reshuffles nearly the whole cluster"

### Consistent hashing with virtual nodes

> **Consistent hashing**: hash both keys *and nodes* onto a circle ([0,1) with wraparound).
> A key belongs to the first node clockwise from it. Adding a node claims only the arc
> between it and its predecessor — everyone else's keys don't move. Each physical node gets
> many **virtual nodes** (vnodes): with only one position per node, arc sizes are wildly
> uneven; many random positions average out, and a node's load spreads over many neighbors.

In [ ]:
class HashRing:
    """Consistent hashing with virtual nodes."""

    def __init__(self, nodes: list[str], vnodes: int = 64):
        self.vnodes = vnodes
        self._points: list[tuple[float, str]] = []   # (ring position, owner) sorted
        for node in nodes:
            self.add_node(node)

    def _positions(self, node: str):
        return [stable_hash(f"{node}#v{i}") / 2**64 for i in range(self.vnodes)]

    def add_node(self, node: str) -> None:
        for pos in self._positions(node):
            self._points.append((pos, node))
        self._points.sort()

    def remove_node(self, node: str) -> None:
        self._points = [(p, n) for p, n in self._points if n != node]

    def owner(self, key: str) -> str:
        """First vnode clockwise from the key's position (wrapping at 1.0)."""
        pos = stable_hash(key) / 2**64
        i = bisect_right(self._points, (pos, chr(0x10FFFF)))
        return self._points[i % len(self._points)][1]

ring = HashRing([f"node{i}" for i in range(5)], vnodes=64)
before = {k: ring.owner(k) for k in keys}
ring.add_node("node5")
after = {k: ring.owner(k) for k in keys}

ch_moved = moved_fraction(before, after)
print(f"consistent hashing, 5 -> 6 nodes: {ch_moved:.1%} of keys move "
      f"(theoretical ideal: 1/6 = {1/6:.1%}; mod-N moved {mod_moved:.0%})")
assert ch_moved < 0.25, "only ~1/N of keys move - rebalancing is now proportional"
# And every moved key moved TO the new node - no pointless churn between old nodes:
assert all(after[k] == "node5" for k in keys if before[k] != after[k])

In [ ]:
# The ring itself, on a small example (3 nodes x 4 vnodes + a few keys):
small = HashRing(["nodeA", "nodeB", "nodeC"], vnodes=4)
fig, ax = viz.hash_ring(
    nodes={n: [stable_hash(f"{n}#v{i}") / 2**64 for i in range(4)]
           for n in ["nodeA", "nodeB", "nodeC"]},
    keys={k: stable_hash(k) / 2**64 for k in ["cart:9", "user:7", "post:3"]},
    title="Arcs = ownership; a key walks clockwise to the next vnode",
)
plt.show()

How many vnodes? More vnodes → smoother arcs → better balance (at the cost of bigger routing
tables). Measure it:

In [ ]:
sample_keys = keys[::5]                       # 6,000 keys is plenty to measure load
vnode_counts = [1, 4, 16, 64, 256]

def avg_imbalance(vnodes: int, clusters: int = 12) -> float:
    """Hottest/average load, averaged over several independent clusters -
    a single cluster is one random draw of arc sizes and can get lucky."""
    ratios = []
    for c in range(clusters):
        ring = HashRing([f"c{c}-node{i}" for i in range(5)], vnodes=vnodes)
        owners = [int(ring.owner(k).split("node")[1]) for k in sample_keys]
        load = np.bincount(owners, minlength=5)
        ratios.append(load.max() / load.mean())
    return float(np.mean(ratios))

imbalance = [avg_imbalance(v) for v in vnode_counts]

fig, ax = plt.subplots()
ax.plot(vnode_counts, imbalance, "o-")
ax.axhline(1.0, color="#9e9e9e", linestyle=":")
ax.set_xscale("log")
ax.set_xlabel("virtual nodes per physical node")
ax.set_ylabel("hottest node ÷ average load")
ax.set_title("Virtual nodes buy balance")
plt.show()

assert imbalance[0] > 1.5, "1 vnode: some node owns a huge arc"
assert imbalance[-1] < 1.15, "256 vnodes: within ~15% of perfect"
assert all(a >= b - 0.1 for a, b in zip(imbalance, imbalance[1:])), "improvement with vnodes"

(Alternatives with the same goal: fixed many-partitions-assigned-to-nodes — create 1,024
partitions up front and move whole partitions between nodes (Kafka, Elasticsearch, Riak) —
or dynamic range splitting like HBase. All are answers to "move little, stay balanced".)

## 3. Skew — when hashing isn't enough

Hashing balances *keys*, not *traffic*. If one key is disproportionately hot — the celebrity
account, the viral post, the whale tenant — its entire load lands on one partition, and no
placement scheme can help, because a single key is indivisible. You must change the key.

**Key salting**: split the hot key into `key#0 ... key#9`, spreading writes over 10
partitions. The cost is symmetric and unavoidable: every *read* of that key must now query
all 10 shards and combine.

In [ ]:
N = 10_000
# A skewed workload: 30% of all writes go to one celebrity's counter.
workload = ["celebrity" if rng.random() < 0.3 else f"user:{int(rng.integers(50_000))}"
            for _ in range(N)]

plain_load = np.bincount([hash_partition(k, N_PARTS) for k in workload],
                         minlength=N_PARTS)

# Salts are chosen at DESIGN TIME, and not naively: random suffixes hash
# wherever they please and can pile onto the same partition (try suffixes
# 0..9 here: five of them collide onto one partition). So we keep only
# suffixes that land on distinct partitions until every partition is covered:
SALTS: list[int] = []
covered: set[int] = set()
candidate = 0
while len(covered) < N_PARTS:
    p = hash_partition(f"celebrity#{candidate}", N_PARTS)
    if p not in covered:
        covered.add(p)
        SALTS.append(candidate)
    candidate += 1

def salted(key: str) -> str:
    """Hot keys get one of the pre-vetted suffixes -> writes spread over all partitions."""
    if key == "celebrity":
        return f"{key}#{SALTS[int(rng.integers(len(SALTS)))]}"
    return key

salted_load = np.bincount([hash_partition(salted(k), N_PARTS) for k in workload],
                          minlength=N_PARTS)

print(f"hottest/average partition load: plain {plain_load.max()/plain_load.mean():.2f}x"
      f" -> salted {salted_load.max()/salted_load.mean():.2f}x")
assert plain_load.max() / plain_load.mean() > 2.0, "one hot key overwhelms its partition"
assert salted_load.max() / salted_load.mean() < 1.25, "salting spreads the hot key"
# The read-side bill: reassembling the celebrity's data now takes |SALTS| sub-key reads.
celebrity_shards = {hash_partition(f"celebrity#{s}", N_PARTS) for s in SALTS}
assert len(celebrity_shards) == N_PARTS, "vetted salts cover every partition"
print(f"reading the celebrity now touches {len(celebrity_shards)} partitions")

Salt only *known-hot* keys (salting everything would make every read a scatter-gather), and
prefer caching for hot *reads* — salting earns its keep for hot *writes*. The celebrity
problem returns at full scale in capstone 23.

## 4. Secondary indexes across partitions

Primary-key lookups route perfectly: hash the key, go. But "find all users in Lisbon" doesn't
mention the partition key at all. Two designs exist, and they trade the *same* cost between
reads and writes:

- **Local index** (document-partitioned; Cassandra, Elasticsearch): each partition indexes
  *its own* documents. Writes stay local (document + index update on one node). Reads must
  **scatter-gather every partition** — Lisbon users could be anywhere.
- **Global index** (term-partitioned; DynamoDB GSIs): the index is its own partitioned
  structure, sharded by the indexed *term* — all of Lisbon's entries live on one index
  partition. Reads touch **one** partition. Writes now fan out: updating a user touches
  their document partition *plus* every affected term partition (usually asynchronously —
  so global indexes are typically *eventually consistent*, nb 05's lag returning in disguise).

In [ ]:
CITIES = ["lisbon", "porto", "berlin", "paris", "oslo", "rome"]

class DataPartition:
    """Holds documents + a LOCAL secondary index over its own documents."""
    def __init__(self):
        self.docs: dict[str, dict] = {}
        self.local_city_index: dict[str, list[str]] = {}

class Cluster:
    def __init__(self, n_parts: int):
        self.parts = [DataPartition() for _ in range(n_parts)]
        # The GLOBAL index: term-partitioned (by hash of city), one list per city.
        self.global_index = [dict() for _ in range(n_parts)]
        self.stats = {"write_rpcs": 0, "read_rpcs": 0}

    def put_user(self, user_id: str, city: str) -> None:
        p = hash_partition(user_id, len(self.parts))
        part = self.parts[p]
        part.docs[user_id] = {"city": city}
        part.local_city_index.setdefault(city, []).append(user_id)
        self.stats["write_rpcs"] += 1                      # document write (local index rides along)
        # Maintaining the global index costs an EXTRA rpc to the term's partition:
        g = hash_partition(city, len(self.parts))
        self.global_index[g].setdefault(city, []).append(user_id)
        self.stats["write_rpcs"] += 1

    def query_city_local(self, city: str) -> list[str]:
        """Local indexes: ask every partition, merge (scatter-gather)."""
        out = []
        for part in self.parts:                            # ALL partitions consulted
            self.stats["read_rpcs"] += 1
            out += part.local_city_index.get(city, [])
        return sorted(out)

    def query_city_global(self, city: str) -> list[str]:
        """Global index: exactly one partition owns this term."""
        g = hash_partition(city, len(self.parts))
        self.stats["read_rpcs"] += 1
        return sorted(self.global_index[g].get(city, []))

cluster = Cluster(N_PARTS)
for i in range(4000):
    cluster.put_user(f"user:{i:05d}", CITIES[int(rng.integers(len(CITIES)))])

cluster.stats["read_rpcs"] = 0
local_answer = cluster.query_city_local("lisbon")
local_cost = cluster.stats["read_rpcs"]

cluster.stats["read_rpcs"] = 0
global_answer = cluster.query_city_global("lisbon")
global_cost = cluster.stats["read_rpcs"]

assert local_answer == global_answer, "both indexes agree on the answer"
print(f"query 'city=lisbon': local index = {local_cost} partition RPCs, "
      f"global index = {global_cost} RPC")
print(f"write cost per user: 1 RPC (local only) vs 2 RPCs (with global index)")
assert local_cost == N_PARTS and global_cost == 1
assert cluster.stats["write_rpcs"] == 2 * 4000, "the global index taxes every write"

Same trade as nb 02's denormalization and nb 03's indexes: **you can pay at write time or at
read time, but you pay.** Local indexes suit write-heavy data with rare secondary queries;
global indexes suit read-heavy secondary access — priced in write fan-out and eventual
consistency.

## 5. Routing: who knows where key K lives?

Somebody must map `key → partition → node`, and stay current as rebalancing moves things.
Three placements of that knowledge, all used in practice:

1. **Any node can route** (gossip; Cassandra): clients connect anywhere; nodes forward.
   No extra infrastructure; every node participates in membership.
2. **A routing tier** (proxy; e.g. `mongos`, Redis Cluster proxies): clients hit the proxy,
   which tracks assignments — often backed by a **coordination service** (ZooKeeper/etcd)
   that stores the authoritative partition map and notifies subscribers on change.
3. **Partition-aware clients**: the client library caches the map and talks straight to the
   right node — fastest path, at the cost of smarter clients and stale-map handling
   ("moved" redirects).

Whichever you choose, the map itself is *replicated, consistency-critical state* — which is
why option 2 leans on consensus systems, the subject of nb 09.

## 6. Exercises

1. **Multi-tenant SaaS.** You shard by `tenant_id` so each tenant's data is co-located (nice
   for per-tenant queries and deletion). Tenant 4711 is a whale: 100× median size and
   traffic. Diagnose what breaks and propose two mitigations with their costs.
2. **Vnode math.** With 1 vnode per node, the largest of N random arcs is roughly
   `ln(N)/N` of the ring (vs the fair `1/N`). Simulate N=10 nodes to estimate the ratio
   largest-arc / fair-share for vnodes ∈ {1, 100} and reconcile with the plot in §2.
3. **Compound keys.** Design the partition/sort key for IoT readings: 100k sensors, 1 write
   per sensor-second, dashboards query one sensor over a time window, and a daily job scans
   everything. Show why partitioning by plain `timestamp` and by plain `hash(full key)` both
   fail first.
4. **Live resharding.** Sketch a zero-downtime plan to split partition P into P1, P2 while
   writes continue. (Hint: nb 05 gave you the tool — a replication log. What order do you
   copy, replay, and cut over?)
5. **Hot reads.** For a viral *read-mostly* post, compare salting vs a cache in front of the
   partition. Why does salting solve the write side but do nothing for a read-side hot spot?

### Solutions

**1.** The whale saturates its partition's node — p99s degrade for every tenant co-located
with it (noisy neighbor), and no rebalancing helps because the shard key is indivisible.
Mitigations: (a) *dedicated placement* — pin whale tenants to their own nodes (a manual
routing-map override; cheap, but a growing whale eventually outgrows one node too);
(b) *sub-shard the whale* — compound key `(tenant_id, hash(entity_id) % k)` for big tenants,
scattering them across k partitions at the price of scatter-gather for that tenant's
cross-entity queries. (This exact pattern — detect the hot entity, change its key — is
salting with a business name.)

**2.** Simulation below. With 1 vnode the hottest node owns ~2.3× its fair share
(ln 10 ≈ 2.3); with 100 vnodes each node's total ownership is an average of 100 arcs —
variance collapses, matching the §2 curve.

**3.** Partition key `sensor_id` (hashed), sort key `timestamp`: dashboard = one partition,
one contiguous range; daily scan = parallel sweep of all partitions (fine for batch, nb 10).
Plain `timestamp` → §1's disaster (all writes on "now"). Plain `hash(sensor‖ts)` → perfect
balance, but the dashboard's time window scatters across the whole cluster.

**4.** (1) Snapshot-copy P's data into P1, P2 while P keeps serving and *logging* changes;
(2) replay P's change log into P1/P2 until they trail by ~nothing (this is nb 05's
follower-catchup, reused); (3) atomically flip the routing map (the coordination service's
one job) so writes go to P1/P2; (4) replay the final tail, then retire P. The only
"downtime" is a brief write pause at the flip — or none, with a short dual-write window
whose duplicates are deduplicated by idempotent apply (nb 11's theme).

**5.** Salting spreads *writes* across sub-keys, but a hot read still needs the whole value —
now from 10 shards (you made reads worse!). A cache absorbs read multiplicity at one layer
up: 1 write to warm it, millions of reads served, invalidation on update (nb 12's derived
data). Hot writes → change the key; hot reads → add a cache.

In [ ]:
# Exercise 2 simulation:
def hottest_over_fair(n_nodes=10, vnodes=1, trials=300):
    ratios = []
    for t in range(trials):
        r = np.random.default_rng(t)
        points = np.sort(r.random(n_nodes * vnodes))
        arcs = np.diff(points, append=points[0] + 1.0)        # wrap-around arc
        owner_load = arcs.reshape(-1).copy()
        # vnode j belongs to node j % n_nodes (random enough for measurement):
        per_node = np.zeros(n_nodes)
        for j, arc in enumerate(owner_load):
            per_node[j % n_nodes] += arc
        ratios.append(per_node.max() * n_nodes)               # ÷ fair share 1/N
    return float(np.mean(ratios))

r1, r100 = hottest_over_fair(vnodes=1), hottest_over_fair(vnodes=100)
print(f"hottest node vs fair share - 1 vnode: {r1:.2f}x, 100 vnodes: {r100:.2f}x")
assert r1 > 1.9, "single vnode: ~ln(N)x imbalance"
assert r100 < 1.25, "100 vnodes: balance within ~25%"
print("solution simulations verified")

## 7. Takeaways

- Partitioning exists to spread **data and load**; it fails through **skew**. Hash for
  balance, range for locality, compound keys for both.
- Rebalancing must move ~`moved data ∝ change in cluster`, not ∝ cluster size: consistent
  hashing (+ vnodes) or fixed-partition schemes; never `hash % N`.
- A single hot key defeats every placement scheme — you must *change the key* (salt hot
  writes) or absorb it a layer up (cache hot reads).
- Secondary indexes re-run the eternal trade: **local** = cheap writes, scatter-gather
  reads; **global** = one-partition reads, fan-out (and usually async) writes.
- The partition map is itself critical replicated state; routing designs differ in who
  caches it and how they learn it changed.

**Further reading.** DDIA ch. 6; Karger et al., *Consistent Hashing and Random Trees*
(1997); the DynamoDB developer guide on adaptive capacity & GSIs (skew and global indexes,
productionized).

**Next:** [07 · Transactions](07-transactions.ipynb) — many clients, one partition,
interleaved reads and writes: what could go wrong? Everything. We build MVCC and watch each
anomaly die.